In [3]:
import requests
import pandas as pd
import numpy as np

# Coordinates for Al-Hasakah, Syria
LATITUDE = 36.5024
LONGITUDE = 40.7476

def fetch_live_2026_forecast():
    # Utilizing your exact verified URL link parameters
    url = f"https://api.open-meteo.com/v1/forecast?latitude={LATITUDE}&longitude={LONGITUDE}&daily=temperature_2m_max,precipitation_sum&hourly=soil_moisture_3_to_9cm,soil_moisture_9_to_27cm&timezone=Europe%2FMoscow"
    response = requests.get(url)
    
    if response.status_code == 200:
        data = response.json()
        
        # 1. Ingest and Resample Hourly Soil Variables (3-9cm and 9-27cm)
        hourly_data = data['hourly']
        hourly_df = pd.DataFrame({
            'time': pd.to_datetime(hourly_data['time']),
            'root_zone_moisture': hourly_data['soil_moisture_3_to_9cm'],
            'deep_soil_moisture': hourly_data['soil_moisture_9_to_27cm']
        })
        
        # Downsample the 24 hourly metrics into a single daily average to match model schema
        daily_soil = hourly_df.resample('D', on='time').mean().reset_index()
        daily_soil.rename(columns={'time': 'date'}, inplace=True)
        
        # 2. Ingest Daily Weather Variables
        daily_data = data['daily']
        daily_weather = pd.DataFrame({
            'date': pd.to_datetime(daily_data['time']),
            'max_temp': daily_data['temperature_2m_max'],
            'precipitation': daily_data['precipitation_sum']
        })
        
        # 3. Join the dataframes together on matching calendar dates
        merged_df = pd.merge(daily_weather, daily_soil, on='date')
        
        # 4. Generate the 14-day rolling precipitation feature required by the trained model
        merged_df['rolling_rain_14d'] = merged_df['precipitation'].rolling(window=14, min_periods=1).sum()
        
        return merged_df
    else:
        raise Exception(f"API Stream Request Failed with Status Code: {response.status_code}")

if __name__ == "__main__":
    print("🛰️ Connecting to live Open-Meteo API stream for Syria...")
    try:
        live_df = fetch_live_2026_forecast()
        
        # Export processed active feature matrices to disk
        live_df.to_csv("live_2026_forecast.csv", index=False)
        print("✅ Success! Live hourly and daily metrics unified in 'live_2026_forecast.csv'")
        print(live_df.head(7)) # Prints a quick preview of the synced days in the console
        
    except Exception as e:
        print(f"❌ Pipeline Execution Error: {e}")


🛰️ Connecting to live Open-Meteo API stream for Syria...
✅ Success! Live hourly and daily metrics unified in 'live_2026_forecast.csv'
        date  max_temp  precipitation  root_zone_moisture  deep_soil_moisture  \
0 2026-07-26      36.8            0.0            0.209167            0.254917   
1 2026-07-27      39.6            0.0            0.207292            0.254000   
2 2026-07-28      42.4            0.0            0.205500            0.253667   
3 2026-07-29      45.1            0.0            0.204000            0.253000   
4 2026-07-30      46.0            0.0            0.202583            0.252583   
5 2026-07-31      45.8            0.0            0.195542            0.240917   
6 2026-08-01      45.4            0.0            0.184792            0.223583   

   rolling_rain_14d  
0               0.0  
1               0.0  
2               0.0  
3               0.0  
4               0.0  
5               0.0  
6               0.0  
